In [5]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plts

drinking_df = pd.read_csv("smoking_driking_dataset_Ver01.csv")
drinking_df

,sex,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,SMK_stat_type_cd,DRK_YN
0,Male,35,170,75,90.0,1.0,1.0,1.0,1.0,120.0,...,126.0,92.0,17.1,1.0,1.0,21.0,35.0,40.0,1.0,Y
1,Male,30,180,80,89.0,0.9,1.2,1.0,1.0,130.0,...,148.0,121.0,15.8,1.0,0.9,20.0,36.0,27.0,3.0,N
2,Male,40,165,75,91.0,1.2,1.5,1.0,1.0,120.0,...,74.0,104.0,15.8,1.0,0.9,47.0,32.0,68.0,1.0,N
3,Male,50,175,80,91.0,1.5,1.2,1.0,1.0,145.0,...,104.0,106.0,17.6,1.0,1.1,29.0,34.0,18.0,1.0,N
4,Male,50,165,60,80.0,1.0,1.2,1.0,1.0,138.0,...,117.0,104.0,13.8,1.0,0.8,19.0,12.0,25.0,1.0,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
991341,Male,45,175,80,92.1,1.5,1.5,1.0,1.0,114.0,...,125.0,132.0,15.0,1.0,1.0,26.0,36.0,27.0,1.0,N
991342,Male,35,170,75,86.0,1.0,1.5,1.0,1.0,119.0,...,84.0,45.0,15.8,1.0,1.1,14.0,17.0,15.0,1.0,N
991343,Female,40,155,50,68.0,1.0,0.7,1.0,1.0,110.0,...,77.0,157.0,14.3,1.0,0.8,30.0,27.0,17.0,3.0,Y
991344,Male,25,175,60,72.0,1.5,1.0,1.0,1.0,119.0,...,73.0,53.0,14.5,1.0,0.8,21.0,14.0,17.0,1.0,N


In [6]:
num_duplicates = drinking_df.duplicated().sum()
num_total = drinking_df.shape[0]
num_unique = num_total - num_duplicates
print(f"Prosent beholdt: {(num_unique / num_total) * 100:.10f}%")
drinking_df_unique = drinking_df.drop_duplicates()

Prosent beholdt: 99.9973773032%


In [7]:
import numpy as np
import pandas as pd

df = drinking_df_unique.copy()

cat_cols = ["sex", "SMK_stat_type_cd", "DRK_YN", "urine_protein", "hear_left", "hear_right"]
no_log = ["SMK_stat_type_cd", "urine_protein", "hemoglobin", "hear_right", "hear_left", "weight", "height", "age"]

df_log = df.copy()

def transform_with_optional_log(df, col, allow_log=True):

    global df_log

    if col not in df.columns:
        return

    data = df[col].dropna()
    if data.empty:
        return

    if col in cat_cols:
        return

    if allow_log and (data > 0).all():
        log_series = np.log10(data)     
        df_log.loc[log_series.index, col] = log_series

for col in df.columns:
    allow_log = (col not in no_log and col not in cat_cols)
    transform_with_optional_log(df, col, allow_log=allow_log)

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

df = df_log.copy()

columns_to_check = [
    "age", "height", "weight", "waistline",
    "sight_left", "sight_right",
    "SBP", "DBP", "BLDS",
    "tot_chole", "HDL_chole", "LDL_chole", "triglyceride",
    "hemoglobin", "serum_creatinine", "SGOT_AST", "SGOT_ALT", "gamma_GTP",
]

# Beregn z-scores per kolonne (NaN ignoreres i sammenligningen under)
z_scores = df[columns_to_check].apply(zscore)

# Behold rader der alle valgte kolonner har |z| <= 3
mask = (np.abs(z_scores) <= 3).all(axis=1)
df_clean = df[mask].copy()

print(f"Rows before: {len(df)}, after outlier removal: {len(df_clean)}")

# df_clean er nå datasettet uten outliers etter z-score-kriteriets

Rows before: 991320, after outlier removal: 877018


In [9]:
import pandas as pd
import numpy as np

# ===============================
# 0) Start fra df_clean (allerede outlier-filtert)
# ===============================
df = df_clean.copy()

# Normaliser kolonnenavn
df.columns = [c.strip().lower() for c in df.columns]

# Konverter til numerisk der mulig
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="ignore")

# Trygg deling (funksjon som takler både Series og skalar)
def safe_div(num, den):
    num = pd.to_numeric(num, errors="coerce")
    if np.isscalar(den):
        den_val = float(den)
        if den_val == 0:
            return pd.Series(np.nan, index=getattr(num, "index", None)) if hasattr(num, "index") else np.nan
        out = np.divide(num, den_val)
        return pd.Series(out, index=num.index) if hasattr(num, "index") else out
    else:
        den = pd.to_numeric(den, errors="coerce")
        num_arr = num.to_numpy() if hasattr(num, "to_numpy") else num
        den_arr = den.to_numpy() if hasattr(den, "to_numpy") else den
        where_mask = (~pd.isna(num_arr)) & (~pd.isna(den_arr)) & (den_arr != 0)
        out = np.full_like(num_arr, np.nan, dtype="float64")
        np.divide(num_arr, den_arr, out=out, where=where_mask)
        return pd.Series(out, index=getattr(num, "index", None)) if hasattr(num, "index") else out

# ===============================
# 1) Feature engineering (uten obese/bmi_cat, og UTEN sight/hear-aggregater)
# ===============================

# BMI
if {"height", "weight"}.issubset(df.columns):
    valid_h = pd.to_numeric(df["height"], errors="coerce") > 0
    df.loc[valid_h, "bmi"] = df.loc[valid_h, "weight"] / ((df.loc[valid_h, "height"]/100.0) ** 2)

# Hemoglobin-kategori (tertiler)
if "hemoglobin" in df.columns:
    try:
        df["hemoglobin_cat"] = pd.qcut(pd.to_numeric(df["hemoglobin"], errors="coerce"), 3,
                                       labels=["low", "middle", "high"])
    except ValueError:
        df["hemoglobin_cat"] = pd.cut(pd.to_numeric(df["hemoglobin"], errors="coerce"), 3,
                                      labels=["low", "middle", "high"])

# Gamma-GTP kategori (kvintiler)
if "gamma_gtp" in df.columns:
    gamma_series = pd.to_numeric(df["gamma_gtp"], errors="coerce")
    try:
        df["gamma_cat"] = pd.qcut(gamma_series, 5, labels=["very_low","low","middle","high","very_high"])
    except ValueError:
        df["gamma_cat"] = pd.cut(gamma_series, 5, labels=["very_low","low","middle","high","very_high"])

# Totalkolesterol som enkel sum (beholdes)
if {"ldl_chole","hdl_chole","triglyceride"}.issubset(df.columns):
    df["total_cholesterol_sum"] = (
        pd.to_numeric(df["ldl_chole"], errors="coerce")
      + pd.to_numeric(df["hdl_chole"], errors="coerce")
      + pd.to_numeric(df["triglyceride"], errors="coerce")
    )

# Lever-ratios + index (mellomregninger droppes senere)
if {"sgot_ast","sgot_alt"}.issubset(df.columns):
    df["sgot_alt"] = safe_div(df["sgot_ast"], df["sgot_alt"])
if {"gamma_gtp","sgot_ast"}.issubset(df.columns):
    df["gamma_gtp_ast"] = safe_div(df["gamma_gtp"], df["sgot_ast"])
if {"sgot_alt","gamma_gtp_ast"}.issubset(df.columns):
    df["liver_function_index"] = df[["sgot_alt","gamma_gtp_ast"]].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Nyre-index
if {"serum_creatinine","urine_protein"}.issubset(df.columns):
    df["kidney_function_index"] = safe_div(df["serum_creatinine"], 1.2) + safe_div(df["urine_protein"], 150)

# Metabolic panel (sum av markører)
met_cols = [c for c in ["hemoglobin","serum_creatinine","sgot_ast","sgot_alt","gamma_gtp"] if c in df.columns]
if met_cols:
    df["metabolic_panel"] = df[met_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

# ===============================
# 2) Dropp råkolonner + eksplisitt uønskede
#    (MEN behold sight/hear + smk_stat_type_cd + drk_yn)
# ===============================
to_drop = [
    # rådata brukt i transformasjoner
    "height","weight","age","waistline",
    "tot_chole","hdl_chole","ldl_chole","triglyceride",
    "sgot_ast","sgot_alt","gamma_gtp","serum_creatinine","urine_protein",
    "hemoglobin",
    # blodtrykk rå
    "sbp","dbp",
    # alder-kategori
    "age_cat",
    # eksplisitt uønskede / redundante
    "tc_hdl","tg_hdl","gamma_gtp_ast","wht_r",
    "bpi","cv_risk","total_cholesterol_friedewald","cardiovascular_health_index",
    # tidligere ønsket fjernet
    "obese","bmi_cat",
    # IKKE dropp disse: sight_left, sight_right, hear_left, hear_right, smk_stat_type_cd, drk_yn, smk_cat
]
df = df.drop(columns=[c for c in to_drop if c in df.columns], errors="ignore")

# ===============================
# 3) Kode kategorier som skal beholdes til tall
# ===============================
# sex (hvis tekst)
if "sex" in df.columns and df["sex"].dtype == "object":
    df["sex"] = df["sex"].str.lower().map({"male":0, "m":0, "female":1, "f":1})

# hemoglobin_cat (ordinal 0..2)
if "hemoglobin_cat" in df.columns and df["hemoglobin_cat"].dtype == "object":
    df["hemoglobin_cat"] = df["hemoglobin_cat"].map({"low":0, "middle":1, "high":2})

# gamma_cat (ordinal 0..4)
if "gamma_cat" in df.columns and df["gamma_cat"].dtype == "object":
    df["gamma_cat"] = df["gamma_cat"].map({"very_low":0, "low":1, "middle":2, "high":3, "very_high":4})

# smk_stat_type_cd: behold som tall (1=never,2=quit,3=smoker) -> skaleres senere
# smk_cat: hvis finnes som tekst, gjør den binær (0/1)
if "smk_cat" in df.columns and df["smk_cat"].dtype == "object":
    df["smk_cat"] = df["smk_cat"].map({"normal":0, "high":1})

# drk_yn: mappe til 0/1 hvis tekst
if "drk_yn" in df.columns and df["drk_yn"].dtype == "object":
    df["drk_yn"] = df["drk_yn"].str.upper().map({"N":0, "Y":1})

# ===============================
# 4) Skaler ALT til [0,1] (inkl. sight/hear + smk_stat_type_cd + drk_yn)
# ===============================
def minmax_scale_01(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    xmin = x.min(skipna=True); xmax = x.max(skipna=True)
    if pd.isna(xmin) or pd.isna(xmax) or xmax == xmin:
        return x.apply(lambda v: np.nan if pd.isna(v) else 0.0)
    return (x - xmin) / (xmax - xmin)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[num_cols] = df[num_cols].apply(minmax_scale_01, axis=0)

# ===============================
# 5) Lagre
# ===============================
out_path = "drinking_df_ready_for_clustering_v3.csv"
df.to_csv(out_path, index=False)
print(f"✅ Saved dataset with sight/hear + smk_stat_type_cd + drk_yn kept and scaled to [0,1] -> {out_path}")


/tmp/ipykernel_19237/2042881176.py:14: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")


✅ Saved dataset with sight/hear + smk_stat_type_cd + drk_yn kept and scaled to [0,1] -> drinking_df_ready_for_clustering_v3.csv


In [ ]:
# === K-MEANS: automatisk valg av k med silhouette ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# -----------------------------
# 1) Last data
# -----------------------------
CSV_PATH = "drinking_df_ready_for_clustering_v3.csv"
df = pd.read_csv(CSV_PATH)
print(f"[INFO] Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")

# -----------------------------
# 2) Klargjør features (utelat røyk/drikk)
# -----------------------------
cols_exclude = ["smk_stat_type_cd", "drk_yn"]
features = [c for c in df.columns if c not in cols_exclude]

X = df[features].select_dtypes(include=[np.number])

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

# Sample for plotting (for hastighet)
plot_n = min(20000, len(X_scaled))
rng = np.random.default_rng(42)
plot_idx = rng.choice(len(X_scaled), size=plot_n, replace=False)
Xs_plot = X_scaled.iloc[plot_idx].to_numpy()

# -----------------------------
# 3) Søk over k og velg beste etter silhouette
# -----------------------------
k_range = range(2, 13)  # juster ved behov
sil_scores = []

best_score = -np.inf
best_k = None
best_labels = None
best_model = None

for k in k_range:
    km = MiniBatchKMeans(
        n_clusters=k, random_state=42, batch_size=10000, n_init=10
    )
    labels = km.fit_predict(X_scaled)
    try:
        sil = silhouette_score(X_scaled, labels)
    except Exception:
        sil = np.nan
    sil_scores.append(sil)
    if np.isfinite(sil) and sil > best_score:
        best_score = sil
        best_k = k
        best_labels = labels
        best_model = km

print(f"[K-Means] Beste k={best_k}, silhouette={best_score:.4f}")

# -----------------------------
# 4) PCA for plotting (2D)
# -----------------------------
pca = PCA(n_components=2, random_state=42)
X2 = pca.fit_transform(Xs_plot)
print("PCA explained variance ratio:", pca.explained_variance_ratio_)

# -----------------------------
# 5) Visualisering
# -----------------------------
# Silhouette vs k
plt.figure(figsize=(6,4))
plt.plot(list(k_range), sil_scores, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette score")
plt.title("K-Means: silhouette vs k")
plt.tight_layout()
plt.show()

# Scatter på PCA
labels_plot_km = best_labels[plot_idx]
plt.figure(figsize=(6,5))
sc = plt.scatter(X2[:,0], X2[:,1], c=labels_plot_km, s=6, alpha=0.7)
plt.title(f"K-Means (k={best_k}) – PCA 2D")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.colorbar(sc, label="cluster")
plt.tight_layout()
plt.show()

# (Valgfritt) enkel oppsummering på røyking/drikking pr klynge hvis kolonnene finnes
if all(col in df.columns for col in ["smk_stat_type_cd", "drk_yn"]):
    summary_km = pd.DataFrame({
        "cluster": best_labels,
        "smk_mean": df["smk_stat_type_cd"],
        "drk_mean": df["drk_yn"]
    }).groupby("cluster").mean(numeric_only=True)
    print("\n=== K-Means cluster summary (mean smoking/drinking levels) ===")
    print(summary_km.round(3))


In [ ]:
# === DBSCAN: automatisk valg av eps og min_samples med silhouette ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# -----------------------------
# 1) Last data
# -----------------------------
CSV_PATH = "drinking_df_ready_for_clustering_v3.csv"
df = pd.read_csv(CSV_PATH)
print(f"[INFO] Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")

# -----------------------------
# 2) Klargjør features (utelat røyk/drikk)
# -----------------------------
cols_exclude = ["smk_stat_type_cd", "drk_yn"]
features = [c for c in df.columns if c not in cols_exclude]

X = df[features].select_dtypes(include=[np.number])

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

# Sample for plotting (for hastighet)
plot_n = min(20000, len(X_scaled))
rng = np.random.default_rng(42)
plot_idx = rng.choice(len(X_scaled), size=plot_n, replace=False)
Xs_plot = X_scaled.iloc[plot_idx].to_numpy()

# -----------------------------
# 3) Grid-søk over eps og min_samples
#    Velg kombo med høyest silhouette (krever >= 2 klynger)
# -----------------------------
eps_list = np.linspace(0.3, 3.0, 15)     # juster rekkevidde ved behov
min_samples_list = [5, 10, 20, 50]       # juster ved behov

best_score = -np.inf
best_params = None
best_labels = None
best_model = None
results = []

for eps in eps_list:
    for ms in min_samples_list:
        db = DBSCAN(eps=eps, min_samples=ms, n_jobs=-1)
        labels = db.fit_predict(X_scaled)

        # Antall klynger (ekskludér støy -1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int(np.sum(labels == -1))

        # Krev minst 2 klynger for gyldig silhouette
        if n_clusters >= 2:
            try:
                sil = silhouette_score(X_scaled[labels != -1], labels[labels != -1])
            except Exception:
                sil = np.nan
        else:
            sil = np.nan

        results.append((eps, ms, n_clusters, n_noise, sil))

        if np.isfinite(sil) and sil > best_score:
            best_score = sil
            best_params = (eps, ms)
            best_labels = labels
            best_model = db

if best_params is None:
    # Fallback: kjør med et konservativt sett parametre
    print("[DBSCAN] Fant ingen kombo med >=2 klynger. Bruker fallback eps=1.0, min_samples=20.")
    best_model = DBSCAN(eps=1.0, min_samples=20, n_jobs=-1).fit(X_scaled)
    best_labels = best_model.labels_
    best_params = (1.0, 20)
    best_score = np.nan

eps_best, ms_best = best_params
n_clusters_best = len(set(best_labels)) - (1 if -1 in best_labels else 0)
n_noise_best = int(np.sum(best_labels == -1))

print(f"[DBSCAN] Beste eps={eps_best:.3f}, min_samples={ms_best}, "
      f"clusters={n_clusters_best}, noise={n_noise_best}, silhouette={best_score if np.isfinite(best_score) else np.nan:.4f}")

# -----------------------------
# 4) PCA for plotting (2D)
# -----------------------------
pca = PCA(n_components=2, random_state=42)
X2 = pca.fit_transform(Xs_plot)
print("PCA explained variance ratio:", pca.explained_variance_ratio_)

# -----------------------------
# 5) Visualisering
# -----------------------------
# Resultatmatrise (eps vs silhouette for en valgt min_samples) – valgfritt
# Vi viser beste min_samples sin kurve for rask inspeksjon
res_df = pd.DataFrame(results, columns=["eps", "min_samples", "n_clusters", "n_noise", "silhouette"])
best_ms = res_df.loc[res_df["silhouette"].idxmax(), "min_samples"] if res_df["silhouette"].notna().any() else min_samples_list[0]
plot_df = res_df[res_df["min_samples"] == best_ms].sort_values("eps")

plt.figure(figsize=(6,4))
plt.plot(plot_df["eps"], plot_df["silhouette"], marker="o")
plt.xlabel("eps")
plt.ylabel("Silhouette score (uten støy)")
plt.title(f"DBSCAN: silhouette vs eps (min_samples={best_ms})")
plt.tight_layout()
plt.show()

# Scatter på PCA
labels_plot_db = best_labels[plot_idx]
plt.figure(figsize=(6,5))
sc = plt.scatter(X2[:,0], X2[:,1], c=labels_plot_db, s=6, alpha=0.7)
plt.title(f"DBSCAN – PCA 2D (noise = -1) | eps={eps_best:.2f}, min_samples={ms_best}")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.colorbar(sc, label="cluster")
plt.tight_layout()
plt.show()

# (Valgfritt) enkel oppsummering på røyking/drikking pr klynge uten støy
if all(col in df.columns for col in ["smk_stat_type_cd", "drk_yn"]):
    mask = best_labels != -1
    if mask.any():
        summary_db = pd.DataFrame({
            "cluster": best_labels[mask],
            "smk_mean": df.loc[mask, "smk_stat_type_cd"],
            "drk_mean": df.loc[mask, "drk_yn"]
        }).groupby("cluster").mean(numeric_only=True)
        print("\n=== DBSCAN cluster summary (mean smoking/drinking levels, uten støy) ===")
        print(summary_db.round(3))

